# ML Coding — Day 10: Quantization IV (QAT & backward)

**~1 hour, vectorized NumPy.** Quantization-aware training: the straight-through estimator, LSQ's
learnable step size, a full QAT step, and stochastic rounding. Backward-heavy — but note the
quantizer's gradient is a **defined convention** (round has zero gradient a.e.), so tests check the STE
/ LSQ formulas directly, not finite differences. No element loops in solutions (a loop over seeds in a
test is fine). Run the **helpers cell first**.

**Q1** STE fake-quant fwd+bwd `[warmup]` · **Q2** LSQ step-size grad `[medium]` · **Q3** per-channel
backward `[medium]` · **Q4** one QAT step `[hard]` · **Q5** stochastic rounding `[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np

## Q1 — STE fake-quant  ·  `[warmup]`

**Paper:** Bengio et al., *STE*, 2013 (arXiv:1308.3432). `fake_quant(x, scale, qmax) =
clip(round(x/scale), -qmax, qmax)·scale` simulates quantization in the forward pass. Its backward uses
the **straight-through estimator**: gradient passes through where the value is in range, and is **zero
where clipped**.

`fake_quant(x, scale, qmax)` and `fake_quant_backward(g, x, scale, qmax) -> dx` (`dx = g` in range,
else 0). No loops.

In [ ]:
import numpy as np


def fake_quant(x, scale, qmax):
    raise NotImplementedError


def fake_quant_backward(g, x, scale, qmax):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    rng = np.random.default_rng(0)
    x = rng.uniform(-2, 2, 50); scale, qmax = 0.1, 8
    assert np.allclose(fake_quant(x, scale, qmax), np.clip(np.round(x / scale), -qmax, qmax) * scale)
    g = rng.standard_normal(50)
    q = np.round(x / scale); inr = (q >= -qmax) & (q <= qmax)
    assert np.allclose(fake_quant_backward(g, x, scale, qmax), g * inr)
    assert (~inr).any()                                  # some values are clipped
    print("Q1 OK")

_q1()

## Q2 — LSQ: learnable step size  ·  `[medium]`

**Paper:** Esser et al., *LSQ*, 2019 (arXiv:1902.08153). **Why it matters:** LSQ makes the quantizer's
**step size a trained parameter**, with a specific gradient. For `q̂ = scale·clip(round(x/scale), qmin,
qmax)`, the gradient w.r.t. scale is `round(v) − v` for in-range `v = x/scale`, `qmin` below, `qmax`
above (a designed STE).

`lsq_forward(x, scale, qmin, qmax)` and `lsq_backward(g, x, scale, qmin, qmax) -> (dx, dscale)`
(`dx` = STE as in Q1; `dscale = Σ g · ∂q̂/∂scale`). No loops.

In [ ]:
def lsq_forward(x, scale, qmin, qmax):
    raise NotImplementedError


def lsq_backward(g, x, scale, qmin, qmax):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(1)
    x = rng.uniform(-1.5, 1.5, 40); scale, qmin, qmax = 0.1, -8, 7
    assert np.allclose(lsq_forward(x, scale, qmin, qmax), np.clip(np.round(x / scale), qmin, qmax) * scale)
    g = rng.standard_normal(40)
    dx, dscale = lsq_backward(g, x, scale, qmin, qmax)
    v = x / scale; r = np.round(v); below = r < qmin; above = r > qmax; inr = ~(below | above)
    assert np.allclose(dx, g * inr)
    gs = np.where(inr, r - v, np.where(below, qmin, qmax))
    assert np.isclose(dscale, np.sum(g * gs))
    print("Q2 OK")

_q2()

## Q3 — Per-channel backward  ·  `[medium]`

Weights are quantized **per output channel** (a scale per row). `per_channel_fake_quant(W, scales,
qmax)` (W `(C, N)`, `scales` `(C,)`, symmetric) and `per_channel_backward(g, W, scales, qmax) -> (dW,
dscale)`: `dW` is the STE mask times `g`; `dscale` is the per-channel LSQ-style gradient summed over
that channel's elements (`round(v) − v` in range, `±qmax` when clipped).

**Lock down:** broadcast `scales[:, None]`; reduce `dscale` along the within-channel axis. No loops.

In [ ]:
def per_channel_fake_quant(W, scales, qmax):
    raise NotImplementedError


def per_channel_backward(g, W, scales, qmax):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(2)
    W = rng.standard_normal((3, 10)) * np.array([[1.], [5.], [20.]])
    scales = np.max(np.abs(W), axis=1) / 7
    g = rng.standard_normal((3, 10))
    dW, dscale = per_channel_backward(g, W, scales, 7)
    assert dW.shape == W.shape and dscale.shape == (3,)
    r = np.round(W / scales[:, None]); inr = (r >= -7) & (r <= 7)
    assert np.allclose(dW, g * inr)
    print("Q3 OK")

_q3()

## Q4 — One QAT training step  ·  `[hard]`

Put it together: fake-quant the weights and activations, run a linear layer + MSE loss, backprop through
the STE, and take one SGD step on the weights. `qat_step(W, x, y, scale_w, scale_x, qmax, lr) ->
(W_new, loss_before, loss_after)`: `pred = fq(x)·fq(W)`, `loss = mean((pred−y)²)`; `dW` uses the STE
mask through the weight quantizer; `W_new = W − lr·dW`. One good step must **reduce** the loss.

**Discuss:** STE lets gradients flow through the non-differentiable quantizer; the latent (full-precision)
weights `W` are what you update; convergence to a quantization-robust solution. No loops.

In [ ]:
def qat_step(W, x, y, scale_w, scale_x, qmax, lr):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(3)
    x = rng.standard_normal((16, 4)); Wtrue = rng.standard_normal((4, 3)); y = x @ Wtrue
    W = rng.standard_normal((4, 3))
    W_new, lb, la = qat_step(W, x, y, scale_w=0.05, scale_x=0.05, qmax=127, lr=0.1)
    assert W_new.shape == W.shape and la < lb            # one QAT step lowers the loss
    print("Q4 OK")

_q4()

## Q5 — Stochastic rounding  ·  `[hard · tensor trick]`

**Why it matters:** deterministic rounding is biased (it always drops the fractional part); **stochastic
rounding** rounds up with probability equal to the fractional part, so it's **unbiased in expectation**
— important for low-precision training/accumulation. `stochastic_round(x, seed) -> array`: round each
`x` to `floor(x)+1` with probability `frac(x)`, else `floor(x)`; use `np.random.default_rng(seed)`.

**The trick:** fully vectorized — `floor + (rng.random(shape) < frac)`. Over many seeds the mean → `x`.
No element loops in the solution.

In [ ]:
def stochastic_round(x, seed):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    x = np.array([0.3, 1.7, 2.5, -0.2])
    out = stochastic_round(x, seed=0)
    assert np.all((out == np.floor(x)) | (out == np.ceil(x)))
    assert np.array_equal(stochastic_round(x, 0), out)   # deterministic per seed
    acc = np.zeros(4)
    for s in range(4000):                                # average over seeds -> unbiased
        acc += stochastic_round(x, seed=s)
    assert np.all(np.abs(acc / 4000 - x) < 0.05)
    print("Q5 OK")

_q5()

## Likely follow-ups
- LSQ's gradient scaling for stability; PACT (learnable clipping); per-group vs per-channel scales.
- QAT vs PTQ — when each wins; folding BatchNorm into the conv before quantizing.
- Stochastic rounding for low-precision optimizer states (8-bit Adam); error feedback in training.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def ref_fake_quant(x, scale, qmax):
    return np.clip(np.round(np.asarray(x, float) / scale), -qmax, qmax) * scale


def ref_fake_quant_backward(g, x, scale, qmax):
    q = np.round(np.asarray(x, float) / scale)
    return np.asarray(g, float) * ((q >= -qmax) & (q <= qmax))


def ref_lsq_forward(x, scale, qmin, qmax):
    return np.clip(np.round(np.asarray(x, float) / scale), qmin, qmax) * scale


def ref_lsq_backward(g, x, scale, qmin, qmax):
    x = np.asarray(x, float); g = np.asarray(g, float)
    v = x / scale
    r = np.round(v)
    below, above = r < qmin, r > qmax
    inr = ~(below | above)
    dx = g * inr
    grad_s = np.where(inr, r - v, np.where(below, qmin, qmax))
    return dx, float(np.sum(g * grad_s))


def ref_per_channel_fake_quant(W, scales, qmax):
    s = np.asarray(scales, float)[:, None]
    return np.clip(np.round(np.asarray(W, float) / s), -qmax, qmax) * s


def ref_per_channel_backward(g, W, scales, qmax):
    W = np.asarray(W, float); g = np.asarray(g, float)
    s = np.asarray(scales, float)[:, None]
    v = W / s
    r = np.round(v)
    inr = (r >= -qmax) & (r <= qmax)
    dW = g * inr
    grad_s = np.where(inr, r - v, np.sign(v) * qmax)
    return dW, np.sum(g * grad_s, axis=1)


def ref_qat_step(W, x, y, scale_w, scale_x, qmax, lr):
    W = np.asarray(W, float); x = np.asarray(x, float); y = np.asarray(y, float)

    def fq(t, s):
        return np.clip(np.round(t / s), -qmax, qmax) * s

    def mask(t, s):
        r = np.round(t / s)
        return (r >= -qmax) & (r <= qmax)

    xq, Wq = fq(x, scale_x), fq(W, scale_w)
    pred = xq @ Wq
    loss_before = np.mean((pred - y) ** 2)
    dpred = 2.0 * (pred - y) / pred.size
    dW = (xq.T @ dpred) * mask(W, scale_w)              # STE through the weight quantizer
    W_new = W - lr * dW
    loss_after = np.mean((fq(x, scale_x) @ fq(W_new, scale_w) - y) ** 2)
    return W_new, loss_before, loss_after


def ref_stochastic_round(x, seed):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    fl = np.floor(x)
    return fl + (rng.random(x.shape) < (x - fl))


_x = np.array([0.0, 0.5, 1.0])
assert np.allclose(ref_fake_quant(_x, 0.5, 4), [0.0, 0.5, 1.0])
_dx, _ds = ref_lsq_backward(np.ones(3), np.array([0.05, 0.4, 100.0]), 0.1, -8, 7)
assert _dx.shape == (3,) and np.isscalar(_ds)
_dW, _dsc = ref_per_channel_backward(np.ones((2, 4)), np.random.default_rng(1).standard_normal((2, 4)),
                                     np.array([0.1, 0.2]), 7)
assert _dsc.shape == (2,)
_rng = np.random.default_rng(0)
_W = _rng.standard_normal((4, 2)); _xx = _rng.standard_normal((8, 4)); _y = _xx @ _rng.standard_normal((4, 2))
_, _lb, _la = ref_qat_step(_W, _xx, _y, 0.05, 0.05, 127, 0.1); assert _la < _lb
assert np.all(np.isin(ref_stochastic_round(np.array([1.3, 2.9]), 0), [1, 2, 3]))
print("reference solutions OK")